# Gathering HLS4ML-reports

Use a environment with HLS4ML and Pandas installed. 

This is based on work done by the group in another course, [DAT255](https://github.com/lolbraa/DAT255-project).

In [7]:
import hls4ml 

search_dir = '../'

patterns = { 
    "All HLS4ML-projects"  : "hls4ml_prj*/",
    #"MNIST HLS4ML-projects" : "MNIST/*/hls4ml_prj*/",  
    #"SVHN HLS4ML-projects"  : "SVHN/*/hls4ml_prj*/",  
}

save_reports = 'export/'

In [ ]:
from pathlib import Path
import sys

#os.listdir('../', )
reports = {}

for label,pattern in patterns.items():
    print(f"Searching {label} ({pattern})")
    for path in Path(search_dir).rglob(pattern):
        if path.is_dir():
            # https://docs.python.org/3/library/pathlib.html#pathlib.PurePath
            #path = path.absolute()
            project_name = f"{path.parts[-3]}-{path.parts[-2]}-{path.name}" # Datasett - Modellarkitektur - Modell revisjon - Navn HLS4ML-prosjektmappe
            print(f"\n\n%%%%%%% {project_name} found in {path} %%%%%%%%%%%\n")
            report_path = f"{save_reports}/{project_name}.rpt"
            with open(report_path, 'w') as f: # TODO: sjekke at det faktisk er innhold, ellers ikke lagre
                sys.stdout = f
                hls4ml.report.read_vivado_report(str(path), full_report=True) # https://github.com/fastmachinelearning/hls4ml/blob/main/hls4ml/report/vivado_report.py
                sys.stdout = sys.__stdout__
                # parse report makes a "summary"
            summary = hls4ml.report.parse_vivado_report(str(path))
            reports.update({
                project_name : {
                    'path' : str(path.absolute()),
                    'summary' : summary,
                    'report_path' : str(report_path)
                    }
                })

Searching All HLS4ML-projects (hls4ml_prj*/)


%%%%%%% jettag-hgq2-Training_FixedHP-hls4ml_prj_VitisUnified_2025 found in ../jettag/jettag-hgq2/Training_FixedHP/hls4ml_prj_VitisUnified_2025 %%%%%%%%%%%

Unable to read project data. Exiting.


%%%%%%% jettag-hgq2-Training_AdaptiveHP-hls4ml_prj_VitisUnified_2025 found in ../jettag/jettag-hgq2/Training_AdaptiveHP/hls4ml_prj_VitisUnified_2025 %%%%%%%%%%%

Unable to read project data. Exiting.


%%%%%%% jet_baseline-the_baseline-hls4ml_prj_VitisUnified found in ../jettag/jet_baseline/the_baseline/hls4ml_prj_VitisUnified %%%%%%%%%%%

Unable to read project data. Exiting.


%%%%%%% MNIST_CNN-2-hls4ml_prj_mnist_hls4ml_VU found in ../MNIST_CNN/2/hls4ml_prj_mnist_hls4ml_VU %%%%%%%%%%%

CSynthesis report not found.
Vivado synthesis report not found.
Cosim report not found.
Timing report not found.


%%%%%%% MNIST_CNN-1-hls4ml_prj_mnist_hls4ml_VU found in ../MNIST_CNN/1/hls4ml_prj_mnist_hls4ml_VU %%%%%%%%%%%

Unable to read project data. Exiting.


Unable to read project data. Exiting.


%%%%%%% testmodel-2-hls4ml_prj_VitisUnifiedKV260_1 found in ../Old_exports_hgq/testmodel/2/hls4ml_prj_VitisUnifiedKV260_1 %%%%%%%%%%%

Unable to read project data. Exiting.


%%%%%%% testmodel-2-hls4ml_prj_VitisUnifiedKV260_2025.2_from_docker found in ../Old_exports_hgq/testmodel/2/hls4ml_prj_VitisUnifiedKV260_2025.2_from_docker %%%%%%%%%%%

Unable to read project data. Exiting.


%%%%%%% testmodel-1-hls4ml_prj_1 found in ../Old_exports_hgq/testmodel/1/hls4ml_prj_1 %%%%%%%%%%%

Cosim report not found.
Timing report not found.


%%%%%%% testmodel-1-hls4ml_prj_VitisUnifiedKV260_1 found in ../Old_exports_hgq/testmodel/1/hls4ml_prj_VitisUnifiedKV260_1 %%%%%%%%%%%

Unable to read project data. Exiting.


%%%%%%% testmodel-1-hls4ml_prj_VitisUnifiedKV260_2 found in ../Old_exports_hgq/testmodel/1/hls4ml_prj_VitisUnifiedKV260_2 %%%%%%%%%%%

Unable to read project data. Exiting.


In [4]:
display(reports)

{}

Extract summary to table

In [ ]:
import pandas as pd

summary_fields = [#'CSynthesisReport.BestLatency', 
                  'CSynthesisReport.LUT', 'CSynthesisReport.FF', 'CSynthesisReport.DSP', 'CSynthesisReport.BRAM_18K']

base_columns = ["project_name"]
summary_rows = [
    {
        "project_name": project_name,
        **report["summary"],
    }
    for project_name, report in reports.items()
]

results_table = pd.json_normalize(summary_rows, sep=".")
summary_columns = [field for field in summary_fields if field in results_table.columns]

resource_columns = {
    "CSynthesisReport.LUT": ["AvailableLUT", "CSynthesisReport.AvailableLUT"],
    "CSynthesisReport.FF": ["AvailableFF", "CSynthesisReport.AvailableFF"],
    "CSynthesisReport.DSP": ["AvailableDSP", "CSynthesisReport.AvailableDSP"],
    "CSynthesisReport.BRAM_18K": ["AvailableBRAM_18K", "CSynthesisReport.AvailableBRAM_18K"],
    #"CSynthesisReport.URAM": ["AvailableURAM", "CSynthesisReport.AvailableURAM"],
}

percentage_columns = []
for used_column, available_candidates in resource_columns.items():
    if used_column not in results_table.columns:
        continue

    available_column = next((column for column in available_candidates if column in results_table.columns), None)
    if available_column is None:
        continue

    used = pd.to_numeric(results_table[used_column], errors="coerce")
    available = pd.to_numeric(results_table[available_column], errors="coerce")
    percentage_column = f"{used_column}.pct"
    results_table[percentage_column] = (used / available) * 100
    percentage_columns.append(percentage_column)

results_table = results_table.loc[:, [*base_columns, *summary_columns, *percentage_columns]]
display(results_table.style.format({column: "{:.0f}%" for column in percentage_columns}))
